## 🧬 LLM × Evolutionary Algorithms

### Learning objectives
- Call an LLM (Gemini) from Python and parse the code it returns.
- Use an LLM to generate metaheuristics and benchmark them on a standard test function.
- Iterate on those algorithms with simple evolutionary operators.

### 1. Environment setup
- Sign in to [Google AI Studio](https://aistudio.google.com/api-keys) and generate an API key.
- We use Gemini for its generous free tier (~1500 requests/day).

In [ ]:
# ! pip install google-genai

In [ ]:
from google import genai

MODEL = "gemini-2.5-flash"
client = genai.Client(api_key="paste-api-key-here")

# Smoke test
print(client.models.generate_content(model=MODEL, contents=["Hello, world!"]).text)

### 2. Generating metaheuristics with an LLM

LLMs can write Python, so we can ask them to write *optimization algorithms*. The workflow is simple: prompt the model with a function signature, parse the code from its response, and run it on a benchmark.

We'll need a small utility to extract a function from a Markdown code block, validate it, and load it into the current namespace:

In [3]:
import ast
import re
from typing import Any, Callable


class FunctionParser:
    """Extracts a Python function from an LLM response and loads it as a callable.

    LLMs typically return code wrapped in a Markdown ```python ... ``` block,
    often surrounded by explanatory prose. This helper isolates the code,
    checks it parses as valid Python, executes it in a fresh namespace, and
    returns the named function so it can be called like any other callable.
    """

    @staticmethod
    def parse(model_response: str, function_name: str) -> Callable | None:
        """Parse `model_response` and return the function named `function_name`.

        Returns None if no code block is found or if the code is syntactically
        invalid. Otherwise the extracted code is executed and the requested
        function is looked up in the resulting namespace.
        """
        function_str = FunctionParser.extract_code(model_response)
        if not function_str or not FunctionParser.validate_function_syntax(function_str):
            return None
        namespace: dict[str, Any] = {}
        # WARNING: exec runs arbitrary model output. In production this MUST
        # happen inside a sandbox (subprocess, container, restricted runtime).
        exec(function_str, namespace)
        return namespace[function_name]

    @staticmethod
    def validate_function_syntax(function_str: str) -> bool:
        """Return True iff `function_str` parses as valid Python (AST check only).

        This catches malformed code before we exec it, but does not guarantee
        the code is safe or semantically correct.
        """
        try:
            ast.parse(function_str)
            return True
        except SyntaxError:
            return False

    @staticmethod
    def extract_code(text: str) -> str | None:
        """Pull the first ```python ... ``` block out of an LLM response.

        Returns the inner code as a string, or None if no fenced Python block
        is present.
        """
        match = re.search(r"```python\s*(.*?)\s*```", text, re.DOTALL)
        return match.group(1) if match else None

Now ask the LLM to invent a metaheuristic. We pin the function signature so the response is directly executable:

In [ ]:
PROMPT_METAHEURISTIC = """
Problem: You are tasked with inventing a novel metaheuristic algorithm capable of minimizing an arbitrary real-valued, 
black-box, single-objective function defined over simple bound constraints.

Write a Python function that implements your algorithm. The function must take exactly these arguments:
- function: Callable[np.ndarray], float] - the objective function to minimise.
- bounds: list[tuple[float, float]] - a list of (lower, upper) pairs delimiting the search space for each dimension.
- budget: int - the total number of objective-function evaluations the algorithm may perform.

The function should return a tuple[float, np.ndarray] containing the best objective value found and the corresponding decision vector.

Your solution should be wrapped in a Markdown Python code block.

```python
import numpy as np 

def new_metaheuristic(
	function: Callable[[np.ndarray], float], 
    bounds: list[tuple[float, float]], 
    budget: int
) -> tuple[float, np.ndarray]:
    ...
```
"""

response = client.models.generate_content(model=MODEL, contents=[PROMPT_METAHEURISTIC])
print(response.text)

Parse it and benchmark on Rastrigin (10D, average over 10 seeds):

In [ ]:
import random
from typing import Callable

import numpy as np


def rastrigin(x: np.ndarray) -> float:
    A: float = 10.0
    return float(A * len(x) + np.sum(x**2 - A * np.cos(2 * np.pi * x)))


class OptimizerVerifier:
    def __init__(
        self,
        budget: int = 10_000,
        dims: int = 10,
        seeds_count: int = 10,
        test_function: Callable = rastrigin,
    ) -> None:
        self.budget = budget
        self.dims = dims
        self.seeds_count = seeds_count
        self.test_function = test_function

    def verify(self, optimizer: Callable) -> float:
        bounds = [(-5, 5) for _ in range(self.dims)]
        results = []
        for seed in range(self.seeds_count):
            np.random.seed(seed)
            random.seed(seed)
            best_val, _ = optimizer(self.test_function, bounds, self.budget)
            results.append(best_val)
        return float(np.mean(results))


metaheuristic = FunctionParser.parse(response.text, "new_metaheuristic")
mean_score_for_rastrigin = OptimizerVerifier().verify(metaheuristic)
mean_score_for_rastrigin

### 3. Evolution of Heuristics

Each generated metaheuristic has a measurable fitness (its score on the benchmark). That means we can apply crossover and mutation operators to *textual* solutions and run an evolutionary loop in the space of Python functions that represent optimization algorithms.

### Exercise 1
Describe the meta‑heuristic generated by Gemini. Does the idea make sense? Is it novel? What are its weaknesses?

### Exercise 2
Generate N = 5 distinct algorithms, benchmark them, and report the best.

> **💡 Tip.** The underlying idea is simple: instead of trying to produce a single perfect algorithm, use the LLM as a generator that samples many different approaches. In practice, we could generate thousands of candidate algorithms and automatically evaluate them. Most of them will likely be poor, but that is acceptable — we only need a few strong candidates to discover an effective baseline. The key insight is that the LLM is valuable not just as a solver, but as a high-throughput source of diverse algorithmic ideas.

![monkeys](./monkeys.jpg)

### Exercise 3
Review [Evolution of Heuristics: Towards Efficient Automatic Algorithm Design Using Large Language Model](https://arxiv.org/pdf/2401.02051) and read **3.4 Prompt Strategies**. Re-implement one of the prompt strategies (M1, M2, or M3). For reference, here's the [GitHub repository](https://github.com/FeiLiu36/EoH) and here's the key [file](https://github.com/FeiLiu36/EoH/blob/main/eoh/src/eoh/methods/eoh/eoh_evolution.py) with all prompts.

<details><summary><b>Hint 1</b></summary><br>Read <code>get_prompt_m2</code> method and adapt it to our case (signature of the function is slightly different, but the idea is the same).</details>


<details><summary><b>Hint 2</b></summary><br>The core idea behind M2 is: "Please identify the main algorithm parameters and assist me in creating a new algorithm with different parameter settings for the provided score function." We should incorporate this instruction directly into the prompt together with the generated metaheuristic.</details>


<details><summary><b>Comment</b></summary><br>In the paper they describe it as "M2: Modify the parameters of one selected heuristic. First, one heuristic is selected from the current population. Then, LLM is prompted to try different parameters in the current heuristic instead of designing a new one." In practice, though, the idea is extremely simple.</details>


<details><summary><b>Hint 3</b></summary><br>Use a prompt like this:<pre>I have one algorithm with its code as follows.
```python
{metaheuristic}
```
Please identify the main algorithm parameters and assist me in creating a new algorithm that has a different parameter settings.
Write a Python function that implements your algorithm. The function must take exactly these arguments:
- function: Callable[np.ndarray], float] - the objective function to minimise.
- bounds: list[tuple[float, float]] - a list of (lower, upper) pairs delimiting the search space for each dimension.
- budget: int - the total number of objective-function evaluations the algorithm may perform.
The function should return a tuple[float, np.ndarray] containing the best objective value found and the corresponding decision vector.
Your solution should be wrapped in a Markdown Python code block.
```python
import numpy as np
def new_metaheuristic(
    function: Callable[[np.ndarray], float],
    bounds: list[tuple[float, float]],
    budget: int
) -> tuple[float, np.ndarray]:
    ...
```
Do not give additional explanations.</pre></details>

Is the mutated algorithm better than the original one? How different is it from the original? Experiment with modifying the prompt and observe how the generated algorithm changes. Can you obtain a significant improvement over the original metaheuristic?

### Exercise 3:
Implement Evolution of Heuristics (or at least part of it). Start simple (pseudocode):
```
current_population = [generate_heuristic()]
for _ in range(10):
   parent = current_population[-1]
   new_solution = M2(parent)
   f_new_solution = verify(new_solution)
   current_population.append(new_solution)
```
Save all generated solutions, preferably using a separate file for each one. Analyse the generated algorithms and identify the most common ideas or design patterns. Plot the solution quality across epochs and investigate whether the iterative process exhibits convergence.

### 4. Recommended Reading
- [AlphaEvolve: A Gemini-powered coding agent for designing advanced algorithms](https://deepmind.google/discover/blog/alphaevolve-a-gemini-powered-coding-agent-for-designing-advanced-algorithms/)
- Shojaee, Parshin, et al. [LLM-SR: Scientific equation discovery via programming with large language models.](https://arxiv.org/abs/2404.18400)
- Romera-Paredes, Bernardino, et al. [Mathematical discoveries from program search with large language models.](https://www.nature.com/articles/s41586-023-06924-6)
- van Stein, Niki, and Thomas Bäck. [Llamea: A large language model evolutionary algorithm for automatically generating metaheuristics.](https://arxiv.org/abs/2405.20132)
- Liu, Fei, et al. [Evolution of heuristics: Towards efficient automatic algorithm design using large language model.](https://arxiv.org/abs/2401.02051)
- van Stein, Niki, et al. [BLADE: Benchmark suite for LLM-driven Automated Design and Evolution of iterative optimisation heuristics](https://arxiv.org/html/2504.20183v1)
- [OpenEvolve](https://github.com/codelion/openevolve)